In [ ]:
import pandas as pd
import torch
from torch_engine import simulate_step_gpu

# 1. Wake up the CUDA cores
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Engine running on: {device}")

# Read in the already calculated Q matrix
Q_df = pd.read_csv("./datasets/messi_data_Q_matrix", index_col=0)

# 2. Map strings to integer indices
state_names = list(Q_df.index)
state_to_idx = {state: i for i, state in enumerate(state_names)}
idx_to_state = {i: state for i, state in enumerate(state_names)}

# 3. Convert the Pandas matrix to a PyTorch Tensor and push it to the GPU
# We use float32 because it is the native, highly optimized format for NVIDIA GPUs
Q_tensor = torch.tensor(Q_df.values, dtype=torch.float32, device=device)

print(f"Q-Tensor Shape: {Q_tensor.shape}")

In [ ]:
import time
from torch_engine import run_monte_carlo_gpu

# Kickoff. 0-0, the Home Team starts with the ball in the center circle.
kickoff_state = "G:0_P:1_Z:14"
start_time = time.time()
implied_odds = run_monte_carlo_gpu(
    kickoff_state,
    device=device,
    state_to_idx=state_to_idx,
    idx_to_state=idx_to_state,
    Q_tensor=Q_tensor,
    n_simulations=10000,
)
end_time = time.time()

print(f"Processed 10,000 matches in {end_time - start_time:.4f} seconds.")
print(f"\n--- MATCH PRICING SPREAD ---")
print(f"Home Win: {implied_odds['Win']:.1%}")
print(f"Draw:     {implied_odds['Draw']:.1%}")
print(f"Away Win: {implied_odds['Loss']:.1%}")